### Imports

In [ ]:
import os
import pandas as pd
import numpy as np
from relaiss import constants
import relaiss as rl
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from alerce.core import Alerce
al = Alerce() 

In [ ]:
!pip install alerce 
from alerce.core import Alerce
al = Alerce()  

### Load data

In [ ]:
csv_path = "/Users/jennakempster-taylor/re-laiss/reference_20k_with_durations.csv" 
csv2_path = "/Users/jennakempster-taylor/re-laiss/reference_20k.csv"

# Load
df = pd.read_csv(csv_path, low_memory=False)
df2 = pd.read_csv(csv2_path, low_memory=False)

print("Shape of duration df:", df.shape)
print("Shape of previous df:", df2.shape)

# Grab our RELAISS features
default_lc_features = constants.lc_features_const.copy()
default_host_features = constants.host_features_const.copy()

# quick look at what is included
print("Default LC features (sample):", default_lc_features)
print("Default host features (sample):", default_host_features)

# look at additional headers of df comapred to df2:
additional = set(df.columns) - set(df2.columns)
print("Added columns in df:", additional)

In [ ]:
# Examine how much data in antares and duration:
total = len(df)
missing = df['antares_duration'].isna().sum()
present = total - missing
print("Amount missing:", missing)
print("Amount present:", present)
print( missing / present * 100,"%" )

Much better than previous data 

In [ ]:
added_cols = ['antares_duration', 'duration_days', 'antares_newest_alert', 'antares_oldest_alert']

missing_summary = (
    df[added_cols].isna().sum().to_frame('Missing')
    .assign(Total=len(df))
    .assign(Percent=lambda x: x['Missing'] / x['Total'] * 100)
)

print(missing_summary)


### Cutting here

In [ ]:
# Now build mask to apply to data to remove things that are not supernovae

mask = df['duration_days'] <= 200

df_200cut = df[mask].copy()
print(f"Cut dataset with objects of over 200 days alert span: {len(df_200cut)} rows (out of {len(df)})")

In [ ]:
USE_HOST = False  # use only light curves initially

def overlap(cols, frame):
    # keep only columns present and drop *_err 
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

lc_cols   = overlap(default_lc_features, df_200cut)
host_cols = overlap(default_host_features, df_200cut) if USE_HOST else []
feature_cols = lc_cols + host_cols

print(f"Found {len(lc_cols)} LC features in filtered data.")
if USE_HOST:
    print(f"Found {len(host_cols)} host features in filtered data.")
print(f"Total candidate features: {len(feature_cols)}")


In [ ]:
from sklearn.impute import KNNImputer

# Ensure all is nnumeric:
numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_200cut[c])]
X = df_200cut[numeric_feature_cols].replace([np.inf, -np.inf], np.nan)
knn_imp = KNNImputer(n_neighbors=5, weights="uniform")
X_imp = knn_imp.fit_transform(X)

print(f"X shape: {X.shape}  -> after KNN impute: {X_imp.shape}")

In [ ]:
iso = IsolationForest(
    n_estimators=300,
    contamination="auto",   # use auto for first run
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

scores   = iso.decision_function(X_imp)  # higher = more normal
raw_pred = iso.predict(X_imp)            # -1 anomaly, 1 normal
anomaly  = (raw_pred == -1).astype(int)  # 1 = anomaly

rank = pd.Series(scores).rank(method="first", ascending=True).astype(int)  # 1 = most anomalous

print("Estimated anomaly rate:", anomaly.mean())

In [ ]:
# Attach to df_filt (same length as scores/anomaly/rank)
df_filt = df_200cut.copy()
df_filt["iso_score"] = np.asarray(scores).ravel()
df_filt["iso_anomaly"] = np.asarray(anomaly).ravel()
df_filt["iso_rank"] = np.asarray(rank).ravel()

# Build 'out' from df_filt to keep lengths consistent
out = df_filt[["ZTFID", "r_duration_above_half_flux", "iso_score", "iso_anomaly", "iso_rank"]].copy()

# Optional context columns (from df_filt!)
context_cols = [c for c in ["t0","r_duration_above_half_flux","g_peak_mag","r_peak_mag","mean_g-r","features_valid"]
                if c in df_filt.columns]

preview = pd.concat(
    [
        df_filt[["ZTFID"] + context_cols].reset_index(drop=True),
        out[["r_duration_above_half_flux","iso_score","iso_anomaly","iso_rank"]].reset_index(drop=True)
    ],
    axis=1
)

display(preview.sort_values("iso_rank").head(10))

### Examine anomalies PCA

Looks a bit off on the sclae so look at normalising beforehand:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Standardise (zero mean, unit variance)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp_df)

# PCA on the scaled data
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

#Plot with colors for anomalies
colors = (df_filt["iso_anomaly"] == -1).astype(int)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1],
            c=colors, cmap="coolwarm", alpha=0.6)
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Isolation Forest anomalies in PCA feature space (standardized)")
plt.show()
print("Explained variance ratio:", pca.explained_variance_ratio_)



In [ ]:
pca_components = pd.DataFrame(
    pca.components_,
    columns=X_imp_df.columns,
    index=['PCA1', 'PCA2']
)
pca_components.T.sort_values('PCA1', ascending=False).head(10)


In [ ]:
# Check variance levels 
pca_full = PCA().fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

for i, v in enumerate(cum_var[:10], 1):
    print(f"PCA{i}: {v:.3f}")

In [ ]:
# If needed: pip install plotly
%pip install plotly

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px

# 1) Standardize features for PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp_df)

# 2) 3D PCA
pca3 = PCA(n_components=3, random_state=42)
X_pca3 = pca3.fit_transform(X_scaled)
evr = pca3.explained_variance_ratio_
title = f"3D PCA (Explained variance: PC1={evr[0]:.2f}, PC2={evr[1]:.2f}, PC3={evr[2]:.2f})"

# 3) Build plotting frame (keep indices aligned!)
plot_df = df_filt.copy()
plot_df["PCA1"] = X_pca3[:, 0]
plot_df["PCA2"] = X_pca3[:, 1]
plot_df["PCA3"] = X_pca3[:, 2]

# groups: anomalies, least anomalous, and the rest
top10_idx = plot_df.sort_values("iso_rank").head(10).index
bottom10_idx = plot_df.sort_values("iso_rank").tail(10).index

group = np.full(len(plot_df), "All")
group[plot_df.index.isin(top10_idx)] = "Top-10 anomalies"
group[plot_df.index.isin(bottom10_idx)] = "Least anomalous 10"
plot_df["group"] = group

# 4) Interactive 3D scatter with hover
fig = px.scatter_3d(
    plot_df,
    x="PCA1", y="PCA2", z="PCA3",
    color="group",
    hover_name="ZTFID",
    hover_data=["iso_rank", "iso_anomaly"],  # add more cols if useful
    title=title,
)
fig.update_traces(marker=dict(size=4))
fig.show()


### Use Alerce API

In [ ]:
client = Alerce()

In [ ]:
top10_df = preview.sort_values("iso_rank").head(10).reset_index(drop=True)


def fetch_quick_summary(oid):
    obj   = client.query_object(oid, format="pandas")          # stats (ra, dec, ndet, first/last mjd, etc.)
    probs = client.query_probabilities(oid, format="pandas")   # lc & stamp classifier probabilities
    mags  = client.query_magstats(oid, format="pandas")        # per-band stats
    return obj, probs, mags

# example
oids = list(top10_df["ZTFID"])  # whatever holds your top 10
summaries = {oid: fetch_quick_summary(oid) for oid in oids}

In [ ]:
# Extract list of object IDs
oids = top10_df["ZTFID"].tolist()
print(oids)

from alerce.core import Alerce
client = Alerce()

for oid in oids:
    print(f"\nFetching data for {oid}...")
    
    # Full light curve (detections + non-detections)
    lc   = client.query_lightcurve(oid, format="pandas")
    dets = client.query_detections(oid, format="pandas")
    nond = client.query_non_detections(oid, format="pandas")
    
    # quick check
    print(f"{oid}: {len(dets)} detections, {len(nond)} non-detections")



### Refinements: cut out galactic plane

In [ ]:
# Look at headers to see what info we have on position
# we have ra and dec

from astropy.coordinates import SkyCoord
import astropy.units as u

transient = SkyCoord(df["ra"], df["dec"], unit="deg")


In [ ]:
# Apply cut to original df (e.g. start again)
df = pd.read_csv(csv_path, low_memory=False)

coords = SkyCoord(ra=df["ra"].values * u.deg,
                  dec=df["dec"].values * u.deg,
                  frame="icrs")

b = coords.galactic.b.deg  # Galactic latitude

# Duration cut
# mask_duration = df["duration_days"].isna() | (df["duration_days"] <= 200)
mask_duration = df["duration_days"] <= 200

# Galactic latitude cut exclude plane |b| < 15 degrees
LAT_CUT = 15.0
mask_lat = np.abs(b) >= LAT_CUT

# Combine both
mask = mask_duration & mask_lat


df_cut = df[mask].copy()

print(f"Filtered dataset: {len(df_cut)} rows (out of {len(df)})")
print(f"Removed due to long duration or near galactic plane: {(~mask).sum()}")


In [ ]:
USE_HOST = False  # use only light curves initially

def overlap(cols, frame):
    # keep only columns present and drop *_err 
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

lc_cols   = overlap(default_lc_features, df_cut)
host_cols = overlap(default_host_features, df_cut) if USE_HOST else []
feature_cols = lc_cols + host_cols

print(f"Found {len(lc_cols)} LC features in filtered data.")
if USE_HOST:
    print(f"Found {len(host_cols)} host features in filtered data.")
print(f"Total candidate features: {len(feature_cols)}")


In [ ]:
from sklearn.impute import KNNImputer

# Ensure all is nnumeric:
numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_cut[c])]
X = df_cut[numeric_feature_cols].replace([np.inf, -np.inf], np.nan)
knn_imp = KNNImputer(n_neighbors=5, weights="uniform")
X_imp = knn_imp.fit_transform(X)

print(f"X shape: {X.shape}  -> after KNN impute: {X_imp.shape}")

In [ ]:
iso = IsolationForest(
    n_estimators=300,
    contamination="auto",   # use auto for first run
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

scores   = iso.decision_function(X_imp)  # higher = more normal
raw_pred = iso.predict(X_imp)            # -1 anomaly, 1 normal
anomaly  = (raw_pred == -1).astype(int)  # 1 = anomaly

rank = pd.Series(scores).rank(method="first", ascending=True).astype(int)  # 1 = most anomalous

print("Estimated anomaly rate:", anomaly.mean())

In [ ]:
# Attach to df_filt (same length as scores/anomaly/rank)
df_fil = df_cut.copy()
df_fil["iso_score"] = np.asarray(scores).ravel()
df_fil["iso_anomaly"] = np.asarray(anomaly).ravel()
df_fil["iso_rank"] = np.asarray(rank).ravel()

# Build 'out' from df_filt to keep lengths consistent
out = df_fil[["ZTFID", "r_duration_above_half_flux", "iso_score", "iso_anomaly", "iso_rank"]].copy()

# Optional context columns (from df_filt!)
context_cols = [c for c in ["t0","r_duration_above_half_flux","g_peak_mag","r_peak_mag","mean_g-r","features_valid"]
                if c in df_fil.columns]

preview = pd.concat(
    [
        df_fil[["ZTFID"] + context_cols].reset_index(drop=True),
        out[["r_duration_above_half_flux","iso_score","iso_anomaly","iso_rank"]].reset_index(drop=True)
    ],
    axis=1
)

display(preview.sort_values("iso_rank").head(10))

In [ ]:
from joblib import dump

ARTIFACTS_PATH = "iforest_artifacts.joblib"
dump(
    {
        "iso": iso, 
        "knn_imp": knn_imp, 
        "numeric_feature_cols": numeric_feature_cols
    },
    ARTIFACTS_PATH
)
print(f"Saved: {ARTIFACTS_PATH}")


### Host features and galactic cut 

In [ ]:
USE_HOST = True  # use only light curves initially

def overlap(cols, frame):
    # keep only columns present and drop *_err 
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

lc_cols   = overlap(default_lc_features, df_cut)
host_cols = overlap(default_host_features, df_cut) if USE_HOST else []
feature_cols = lc_cols + host_cols

print(f"Found {len(lc_cols)} LC features in filtered data.")
if USE_HOST:
    print(f"Found {len(host_cols)} host features in filtered data.")
print(f"Total candidate features: {len(feature_cols)}")


# Ensure all is nnumeric:
numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_cut[c])]
X = df_cut[numeric_feature_cols].replace([np.inf, -np.inf], np.nan)
knn_imp = KNNImputer(n_neighbors=5, weights="uniform")
X_imp = knn_imp.fit_transform(X)

print(f"X shape: {X.shape}  -> after KNN impute: {X_imp.shape}")

iso = IsolationForest(
    n_estimators=300,
    contamination="auto",   # use auto for first run
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

scores   = iso.decision_function(X_imp)  # higher = more normal
raw_pred = iso.predict(X_imp)            # -1 anomaly, 1 normal
anomaly  = (raw_pred == -1).astype(int)  # 1 = anomaly

rank = pd.Series(scores).rank(method="first", ascending=True).astype(int)  # 1 = most anomalous

print("Estimated anomaly rate:", anomaly.mean())

# Attach to df_filt (same length as scores/anomaly/rank)
df_fil = df_cut.copy()
df_fil["iso_score"] = np.asarray(scores).ravel()
df_fil["iso_anomaly"] = np.asarray(anomaly).ravel()
df_fil["iso_rank"] = np.asarray(rank).ravel()

# Build 'out' from df_filt to keep lengths consistent
out = df_fil[["ZTFID", "r_duration_above_half_flux", "iso_score", "iso_anomaly", "iso_rank"]].copy()

# Optional context columns (from df_filt!)
context_cols = [c for c in ["t0","r_duration_above_half_flux","g_peak_mag","r_peak_mag","mean_g-r","features_valid"]
                if c in df_fil.columns]

preview = pd.concat(
    [
        df_fil[["ZTFID"] + context_cols].reset_index(drop=True),
        out[["r_duration_above_half_flux","iso_score","iso_anomaly","iso_rank"]].reset_index(drop=True)
    ],
    axis=1
)

display(preview.sort_values("iso_rank").head(10))

### Use Alerce API and pull 100 SN-like objects, run iso forest on these objs

### Look at PCA 'anomalies' and compare

### Test on ~200 transients from Alerce (without Host first)

In [ ]:
from joblib import load
from alerce.core import Alerce
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
# load artifacts from training ( USE_HOST=False) 
art = load("iforest_artifacts.joblib")
iso = art["iso"]
knn_imp = art["knn_imp"]
numeric_feature_cols = art["numeric_feature_cols"]  # must match training order

In [ ]:
from alerce.core import Alerce
alerce = Alerce()
# list classifiers
classifiers = alerce.query_classifiers(format="pandas")  # or json
print(classifiers)

# list classes for one classifier
classes = alerce.query_classes("lc_classifier", format="pandas")
print(classes)

In [ ]:
# fetch 1000 most recent "Transient" objects 
alerce = Alerce()
objs = alerce.query_objects(
    classifier="lc_classifier_top",
    class_name="Transient",
    probability=0.7,
    page_size=1000,
    order_by="lastmjd",
    order_mode="DESC",
    format="pandas"
).drop_duplicates("oid")

#objs = objs.drop_duplicates(subset="oid").reset_index(drop=True)
oids = objs["oid"].tolist()

In [ ]:
probs_stamp = alerce.query_probabilities(
    oids,                 # positional: list of OIDs
    "stamp_classifier",   # positional: classifier name
    "pandas"              # positional: output format
)

# 3) Pivot to wide and threshold on SN probability
wide = probs_stamp.pivot_table(
    index="oid", columns="class_name", values="probability", aggfunc="max"
).fillna(0.0)

# ensure SN column exists
if "SN" not in wide.columns:
    wide["SN"] = 0.0

wide["P_SN"] = wide["SN"]
cand = wide[wide["P_SN"] >= 0.7].index

# 4) keep only those OIDs (SNe by stamp classifier)
sn_objs = objs_top[objs_top["oid"].isin(cand)].copy()
print(f"Total Transients: {len(objs_top)}  |  Likely SNe (stamp P>=0.7): {len(sn_objs)}")
sn_objs.head()


print(f"Total objects returned: {len(objs_top)}")
print(f"Likely SNe (P_SN ≥ 0.7): {len(sn_objs)}")
sn_objs.head()

In [ ]:
# pull feature tables for those OIDs 
def fetch_features(oids, keep_placeholders=True):
    rows, bad = [], []
    for oid in tqdm(oids, desc="Fetching ALeRCE features"):
        try:
            f = alerce.query_features(oid, format="pandas")
            if (f is None) or f.empty or not {"oid","name","value"}.issubset(f.columns):
                if keep_placeholders:
                    # one placeholder row; we'll add missing cols later
                    rows.append(pd.DataFrame(index=pd.Index([oid], name="oid")))
                else:
                    bad.append((oid, "empty-or-missing-cols"))
                continue
            fw = f.pivot_table(index="oid", columns="name", values="value", aggfunc="first")
            rows.append(fw)
        except Exception as e:
            if keep_placeholders:
                rows.append(pd.DataFrame(index=pd.Index([oid], name="oid")))
            else:
                bad.append((oid, str(e)))
    out = pd.concat(rows, axis=0) if rows else pd.DataFrame()
    return out, bad

df_new, bad = fetch_features(oids, keep_placeholders=True)

# Ensure every training column exists, then order them
missing = [c for c in numeric_feature_cols if c not in df_new.columns]
for c in missing:
    df_new[c] = np.nan
X_new = df_new[numeric_feature_cols].replace([np.inf, -np.inf], np.nan)

# guardrails: drop rows that are *entirely* NaN if you prefer
# X_new = X_new.loc[~X_new.isna().all(axis=1)]

print(f"ALeRCE rows fetched: {len(df_new)}  usable after drop: {len(X_new)}  bad: {len(bad)}")


In [ ]:
# apply same imputer and model from training
X_new_imp = knn_imp.transform(X_new)
scores_new = iso.decision_function(X_new_imp)  # higher = more "normal"
pred_new   = iso.predict(X_new_imp)            # -1 outlier, +1 inlier
anom_new   = (pred_new == -1).astype(int)

In [ ]:
# package results
res = pd.DataFrame({
    "oid": df_new.index,
    "iso_score": scores_new,
    "iso_anomaly": anom_new
}).reset_index(drop=True)

In [ ]:
# attach some handy object metadata
keep_cols = [c for c in ["oid","meanra","meandec","ndet","firstmjd","lastmjd","class_name"] if c in objs.columns]
res = res.merge(objs[keep_cols], on="oid", how="left")


### rank by anomaly (lowest score = most anomalous)
res["iso_rank"] = pd.Series(res["iso_score"]).rank(method="first", ascending=True).astype(int)

res.sort_values("iso_rank").to_csv("alerce_200_recent_iforest.csv", index=False)
print("Saved: alerce_200_recent_iforest.csv")
res.sort_values("iso_rank").head(10)
